# Scientific computing in the browser

This notebook shows a typical scientific Python workflow running entirely in
your browser via WebAssembly — no server, no installation required.

NumPy, SciPy, and Matplotlib are pre-installed in the kernel environment from
[emscripten-forge](https://emscripten-forge.org/). They're the same packages
you'd use on your laptop, compiled to WASM instead of native code.

*The signal processing examples below are adapted from the
[SciPy documentation](https://docs.scipy.org/doc/scipy/tutorial/signal.html)
(BSD 3-Clause license) and
[NumPy documentation](https://numpy.org/doc/stable/reference/routines.fft.html)
(BSD 3-Clause license).*

## Signal processing with SciPy

Let's generate a noisy signal, apply a Butterworth low-pass filter, and
visualize the result — a classic DSP workflow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

rng = np.random.default_rng(42)
fs = 1000  # sampling rate (Hz)
t = np.arange(0, 1.0, 1.0 / fs)

# Two sine waves buried in noise
clean = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 12 * t)
noisy = clean + 0.8 * rng.standard_normal(len(t))

# 4th-order Butterworth low-pass at 20 Hz
sos = signal.butter(4, 20, btype="low", fs=fs, output="sos")
filtered = signal.sosfilt(sos, noisy)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, data, title, color in zip(
    axes,
    [clean, noisy, filtered],
    ["Original signal (5 Hz + 12 Hz)", "With noise", "After Butterworth low-pass (20 Hz cutoff)"],
    ["steelblue", "gray", "tomato"],
):
    ax.plot(t[:200], data[:200], color=color, linewidth=1.2)
    ax.set_title(title)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Signal filtering — SciPy in WebAssembly", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Linear algebra

Eigenvalue decomposition of a symmetric matrix — the same LAPACK routines,
compiled to WASM.

In [ ]:
from scipy import linalg

A = rng.standard_normal((5, 5))
A = A + A.T  # make symmetric

eigenvalues, eigenvectors = linalg.eigh(A)

# Verify: A @ v = λ * v
for i, (val, vec) in enumerate(zip(eigenvalues, eigenvectors.T)):
    residual = np.linalg.norm(A @ vec - val * vec)
    print(f"λ_{i} = {val:+8.4f}  |Av - λv| = {residual:.2e}")

print(f"\nAll eigenvalue equations satisfied (residuals < 1e-14)")

## Curve fitting

Non-linear least squares fitting using `scipy.optimize.curve_fit`.

In [ ]:
from scipy.optimize import curve_fit

def damped_sine(t, amp, freq, decay, phase):
    return amp * np.sin(2 * np.pi * freq * t + phase) * np.exp(-decay * t)

t_data = np.linspace(0, 4, 200)
true_params = (2.5, 3.0, 1.2, 0.5)
y_true = damped_sine(t_data, *true_params)
y_noisy = y_true + 0.3 * rng.standard_normal(len(t_data))

popt, pcov = curve_fit(damped_sine, t_data, y_noisy, p0=[2, 2.5, 1, 0])
y_fit = damped_sine(t_data, *popt)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(t_data[::3], y_noisy[::3], s=10, alpha=0.5, label="Noisy data", color="gray")
ax.plot(t_data, y_true, "--", color="steelblue", linewidth=1.5, label="True signal")
ax.plot(t_data, y_fit, color="tomato", linewidth=2, label="Fitted curve")
ax.legend()
ax.set(xlabel="Time (s)", ylabel="Amplitude", title="Non-linear curve fitting — scipy.optimize")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

labels = ["amplitude", "frequency", "decay", "phase"]
print("Parameter      True    Fitted")
for name, true, fitted in zip(labels, true_params, popt):
    print(f"  {name:<12s} {true:6.3f}   {fitted:6.3f}")

## Install additional packages

Need a package that's not pre-installed? Use `%conda install` — it works just
like on your laptop, but runs entirely in your browser.

In [ ]:
%load_ext conda_emscripten

In [ ]:
%conda install lz4

In [ ]:
import lz4.frame

# Compress our fitted curve data for efficient storage
data = y_fit.tobytes()
compressed = lz4.frame.compress(data)
print(f"Array data: {len(data):,} bytes → {len(compressed):,} bytes "
      f"({len(compressed)/len(data)*100:.0f}%)")

roundtrip = np.frombuffer(lz4.frame.decompress(compressed), dtype=np.float64)
print(f"Round-trip matches: {np.array_equal(y_fit, roundtrip)}")

## What's available

All packages on [emscripten-forge](https://emscripten-forge.org/) work here.
Use `%conda search` to explore:

```
%conda search scipy
%conda search scikit-learn
%conda search sympy
```

For the full list, visit [emscripten-forge.org](https://emscripten-forge.org/).